In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from marine_qc import (
    combine_qc_results,
    do_aground_check,
    do_multiple_sequential_check,
    do_speed_check,
    do_sst_biased_check,
)

In [ ]:
from data import get_buoy_data

# How to use quality control checks with sequential reports from buoys

We need some text!!!

In [ ]:
data = get_buoy_data()
data

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 8))

axs[0].plot(data["lon"], data["lat"], marker="o", linestyle="-")
axs[0].set_title("Buoy drifter – latitude/longitude track")
axs[0].set_xlabel("Longitude (°)")
axs[0].set_ylabel("Latitude (°)")
axs[0].grid(True)

axs[1].plot(data["sst"], marker="o", linestyle="-")
axs[1].set_title("Buoy drifter – sea surface temperature")
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Temperature (°C)")
axs[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
qc_speed = do_speed_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    speed_limit=2.5,
    min_win_period=0.5,
    max_win_period=1.0,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "qc_speed": qc_speed})

In [ ]:
qc_aground = do_aground_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    smooth_win=3,
    min_win_period=1,
    max_win_period=2,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "qc_aground": qc_aground})

In [ ]:
ostia = pd.Series([22.0, 21.6, 21.2, 20.8, 20.4, 20.0, 19.6, 19.2, 18.8, 18.4, 18.0, 17.6, 17.2, 16.8, 16.4, 16.0, 16.0, 16.0, 16.0])
qc_biased = do_sst_biased_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    sst=data.sst,
    ostia=ostia,
    ice=[0.0] * 19,
    bgvar=[0.01] * 19,
    n_eval=9,
    bias_lim=0.5,
    drif_intra=1.0,
    drif_inter=0.29,
    err_std_n=3.0,
    n_bad=2,
    background_err_lim=0.3,
)
pd.DataFrame({"date": data.date, "lat": data.lat, "lon": data.lon, "sst": data.sst, "qc_biased": qc_biased})

In [ ]:
qc_dict = {
    "speed_check": {
        "func": "do_speed_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "speed_limit": 2.5,
            "min_win_period": 1,
            "max_win_period": 2,
        },
    },
    "aground_check": {
        "func": "do_aground_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "smooth_win": 3,
            "min_win_period": 1,
            "max_win_period": 2,
        },
    },
}

In [ ]:
qc_multi = do_multiple_sequential_check(
    data,
    qc_dict=qc_dict,
    groupby="buoy_id",
    return_method="failed",
)
qc_multi

In [ ]:
qc_flag = combine_qc_results(qc_multi)
pd.DataFrame({**data, "qc_flag": qc_flag})